In [ ]:
## -*- coding: utf-8 -*-
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from pandas import DataFrame, Series
from math import *
import seaborn as sns
import matplotlib.cm as cm
import collections
import time
from Bio import SeqIO
import re
import itertools
import random
import pickle
import RNA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import multivariate_normal
from scipy.stats import norm
from scipy.stats import chi2
from scipy.spatial import distance
from scipy.sparse import coo_matrix
import warnings
from scipy.spatial.distance import jensenshannon
import subprocess
from numba import njit, prange
import copy
mylen = np.vectorize(len)

warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
### Define functions

'''
Data sampling function

- sampling_dataframe: samples sequences from each round of a dataset according to their counts.
    - N: number of sequences to sample from each round.
    - random_state: seed for reproducibility.

- dataloader: loads the dataset based on the specified name, applies filtering based on length and rounds, and returns the processed dataset.
    - name: name of the dataset to load (e.g., 'N14', 'N20', etc.).
    - length_range: tuple specifying the minimum and maximum length of sequences to include (default is 'default', which applies predefined length filters based on the dataset).
    - N: number of sequences to sample from each round (default is 100000).
    - remove_rounds: list of rounds to remove from the dataset (default is 'default', which applies predefined round removals based on the dataset).
    - method: method for sampling sequences ('all' for random sampling, 'top' for top N sequences by count, 'none' for no sampling).
'''

def sampling_dataframe(ds, N, random_state=None):
    np.random.seed(random_state)
    ds_sampled = []
    for i in range(len(ds)):
        d = ds[i]
        seqs = d['seqs'].values
        counts = d['counts'].values
        if counts.sum() < N:
            print('Not enough counts in round {}: {}'.format(i+1, counts.sum()))
            ds_sampled.append(d)
            continue
        inds = np.random.choice(len(counts), size=N, replace=True, p=counts/counts.sum())
        seqs_sampled = [seqs[i] for i in inds]
        c = collections.Counter(seqs_sampled)
        seqs, counts = zip(*c.most_common())
        ds_sampled.append(DataFrame({'seqs': seqs, 'counts': counts, 'lens': mylen(seqs)}))
    return ds_sampled

def dataloader(name, length_range='default', N=100000, remove_rounds='default', random_state=None, method='all'):
    if name not in ['N14', 'N14_new', 'N14p', 'N14p_new', 'N20', 'N20p', 'single_N14', 'single_N14p', '2mer']:
        raise ValueError(f'Invalid name: {name}. Must be one of ["N14", "N14_new", "N14p", "N14p_new", "N20", "N20p", "single_N14", "single_N14p", "2mer"].')

    if name == 'N20':
        Rs = [2, 4, 6, 8, 11, 13, 15, 17, 19, 22, 24, 26, 28, 29, 30]

    elif name == 'N20p':
        Rs = [1, 2, 4, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]

    elif (name == 'N14') or (name == 'N14p') or (name == 'N14p_new'):
        Rs = [1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]

    elif name == 'N14_new':
        Rs = [0, 1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]

    elif name == 'single_N14' or name == 'single_N14p':
        Rs = [1, 2, 3, 4, 5, 6]

    elif name == '2mer':
        Rs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

    if name == 'single_N14':
        with open(f'data/filtered/Series/single_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)
        ds = ds[:6]
    elif name == 'single_N14p':
        with open(f'data/filtered/Series/single_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)
        ds = ds[6:]
    elif name == '2mer':
        with open(f'data/filtered/Series/2mer_filtered_blast_no_repeats_norm.pkl', 'rb') as f:
            ds = pickle.load(f)
        ds = ds[1:5] + ds[6:10] + ds[11:]
    else:
        with open(f'data/filtered/Series/{name}_filtered_blast.pkl', 'rb') as f:
            ds = pickle.load(f)

    if remove_rounds != 'default':
        if not (isinstance(remove_rounds, (list, tuple)) and all(r in Rs for r in remove_rounds)):
            raise ValueError(f'remove_rounds must be a list or tuple of integers in {Rs}.')
        remove_rounds = np.sort(list(set(remove_rounds)))
        remove_indices = [Rs.index(r) for r in remove_rounds]
        ds = [d for i, d in enumerate(ds) if i not in remove_indices]
        Rs = [r for r in Rs if r not in remove_rounds]
    else:
        if name == 'N20':
            ds = ds[:-1]
            Rs = Rs[:-1]
        elif name == 'N20p':
            ds = ds[1:5] + ds[6:11] + ds[12:17]
            Rs = Rs[1:5] + Rs[6:11] + Rs[12:17]
        elif name == 'N14p':
            ds = ds[:6] + ds[7:12] + ds[13:18]
            Rs = Rs[:6] + Rs[7:12] + Rs[13:18]
        elif name == 'N14_new':
            ds = ds[1:]
            Rs = Rs[1:]

    if length_range != 'default':
        if not (isinstance(length_range, (list, tuple)) and len(length_range) == 2):
            raise ValueError('length_range must be a list or tuple of two integers.')
        min_len, max_len = length_range
        if not (isinstance(min_len, int) and isinstance(max_len, int) and min_len <= max_len):
            raise ValueError('length_range must contain two integers where the first is less than or equal to the second.')
        ds = [d[(d['lens'] >= min_len) & (d['lens'] <= max_len)].reset_index(drop=True) for d in ds]
    else:
        if name in ['N14', 'N14_new', 'N14p', 'N14p_new', 'single_N14', 'single_N14p']:
            ds = [d[(d['lens'] >= 18) & (d['lens'] <= 30)].reset_index(drop=True) for d in ds]
        elif name in ['N20', 'N20p']:
            ds = [d[(d['lens'] >= 25) & (d['lens'] <= 45)].reset_index(drop=True) for d in ds]
        elif name == '2mer':
            ds = [d[(d['lens'] > 22) & (d['lens'] <= 30)].reset_index(drop=True) for d in ds]
    
    if method == 'all':
        ds = sampling_dataframe(ds, N, random_state=random_state)
    elif method == 'top':
        ds = [d.nlargest(N, 'counts').reset_index(drop=True) if len(d) >= N else d for d in ds]
    elif method == 'none':
        pass
    else:
        raise ValueError('method must be "all", "top", or "none".')

    return ds, Rs


'''
Functions for DataFrame processing

- concatenate_dataframes: concatenates a list of DataFrames into a single DataFrame and adds a 'label' column indicating the original round of each sequence.
    - ds: list of DataFrames to concatenate.

- deploy_df_by_counts: expands a DataFrame by replicating each row according to the value in the 'counts' column, effectively "deploying" the DataFrame based on counts.
    - df: DataFrame to deploy.
    - col_name: name of the column containing the counts (default is 'counts').
'''

def concatenate_dataframes(ds):
    lens = [len(d) for d in ds]
    inds_rounds = [np.repeat(i, lens[i]) for i in range(len(ds))]
    d = pd.concat(ds, ignore_index=True)
    d['label'] = np.concatenate(inds_rounds)
    return d

def deploy_df_by_counts(df, col_name='counts'):
    if col_name not in df.columns:
        raise KeyError(f'Column {col_name} not found in DataFrame.')
    counts = df[col_name].values
    if counts.sum() == 0:
        raise ValueError('Sum of counts is zero, cannot deploy DataFrame.')
    df = df.drop(columns=[col_name])
    values = np.zeros((counts.sum(), df.shape[1]), dtype=object)
    start = 0
    for i in range(len(counts)):
        end = start + counts[i]
        values[start:end] = df.iloc[i].values
        start = end
    return DataFrame(values, columns=df.columns)


'''
Functions for BLAST

- make_fasta: creates a FASTA file from a list of sequences.
    - seqs: list of sequences to include in the FASTA file.
    - Name: name of the FASTA file to create (without extension).
    - key: type of sequences ('RNA' or 'DNA') to determine whether to replace 'T' with 'U'.

- read_fasta: reads sequences from a FASTA file and returns them as a list.
    - path: path to the FASTA file to read.

- run_blast: creates a BLAST database from the provided sequences and runs BLAST to compare query sequences against the database.
    - query_seqs: list of sequences to use as queries in the BLAST search.
    - db_seqs: list of sequences to include in the BLAST database.
    - output_name: name of the output file to save BLAST results (without extension).
    - max_target_seqs: maximum number of target sequences to report for each query (default is 5).
    - num_threads: number of threads to use for BLAST (default is 1).
    - use_precomputed_db: whether to use a precomputed BLAST database (default is False).
    - db_name: name of the BLAST database to create or use (default is 'blast_db').
    - rm_db: whether to remove the BLAST database after running BLAST (default is False).

- read_blast_results: reads BLAST results from a TSV file and returns them as a DataFrame.
    - file_name: name of the BLAST results file to read (without extension).

- eval_blast_result: calculates query coverage and identity from BLAST results.
    - df_query: DataFrame containing the original query sequences and their lengths.
    - df_blast: DataFrame containing the BLAST results for the query sequences.
'''

def make_fasta(seqs, Name, key):
    if key=='RNA':
        seqs=[seq.replace('T', 'U') for seq in seqs]
    elif key=='DNA':
        seqs=seqs
    else:
        raise KeyError('Key must be "RNA" or "DNA".')
    seq_name=Name.rsplit('/')[-1]
    seq_list = []
    for i, seq in enumerate(np.array(seqs)):
        seq_list.append('>'+seq_name+'_'+str(i+1)+'\n')
        seq_list.append(seq + '\n')
    path = Name+'.fasta'
    open(path, "w").writelines(seq_list)

def read_fasta(path):
    seqs = []
    for record in SeqIO.parse(path, "fasta"):
        seqs.append(str(record.seq))
    return seqs

def run_blast(query_seqs, db_seqs, output_name, max_target_seqs=5, num_threads=1, use_precomputed_db=False, db_name='blast_db', rm_db=False):
    if not use_precomputed_db:
        make_fasta(db_seqs, db_name, key='DNA')
        subprocess.run(['mkdir', 'blast_db'], stdout=subprocess.DEVNULL)
        subprocess.run(['makeblastdb', '-in', db_name+'.fasta', '-dbtype', 'nucl', '-out', 'blast_db/'+db_name], check=True, stdout=subprocess.DEVNULL)
    make_fasta(query_seqs, 'query', key='DNA')
    subprocess.run(['blastn', '-query', 'query.fasta', '-task', 'blastn', '-db', 'blast_db/'+db_name, '-outfmt', '6', '-strand', 'plus',
                    '-max_target_seqs', str(max_target_seqs), '-out', output_name+'.tsv', '-num_threads', str(num_threads)], check=True, stdout=subprocess.DEVNULL)
    
    subprocess.run(['rm', 'query.fasta'], check=True)
    if not use_precomputed_db:
        subprocess.run(['rm', db_name+'.fasta'], check=True)
    if rm_db:
        subprocess.run(['rm', '-r', 'blast_db'], check=True)

def read_blast_results(file_name):
    columns=['query', 'subject_id', 'ident', 'align_len', 'mismatche', 'gap',
            'q_start', 'q_end', 's_start', 's_end', 'evalue', 'bit_score']
    df_blast = pd.read_csv(file_name+'.tsv', sep='\t', names=columns)
    return df_blast

def eval_blast_result(df_query, df_blast):
    seqs = df_query['seqs'].values
    lens = df_query['lens'].values
    qcover, ident = np.zeros(len(seqs)), np.zeros(len(seqs))
    query_ids = np.array([int(query.split('_')[-1]) for query in df_blast['query'].values])

    c = collections.Counter(query_ids)
    inds = []
    n = 0
    for v in c.values():
        inds += [n]
        n += v
    df_blast = df_blast.iloc[inds].reset_index(drop=True)
    new_inds = np.array(list(c.keys())) - 1
    new_inds = new_inds.astype(int)

    qcover[new_inds] = (df_blast['q_end'].values - df_blast['q_start'].values + 1) / lens[new_inds]
    qcover = qcover * 100
    ident[new_inds] = df_blast['ident'].values
    
    df = pd.concat([df_query, DataFrame({'qcover': qcover, 'ident': ident})], axis=1)
    return df


'''
Functions for k-mer analysis

- make_kmers: generates all possible k-mers from a given set of bases.
    - bases: list of bases to use for generating k-mers
    - k: length of the k-mers to generate.

- normalize: normalizes the k-mer frequency data using the specified method.
    - X: DataFrame containing the k-mer frequencies to normalize.
    - method: method for normalization ('standard' for standard scaling, 'center' for centaring, None for no normalization).

- cal_kmer_freqs: calculates k-mer frequencies for a list of sequences.
    - sequences: list of sequences to analyze.
    - k: length of the k-mers to calculate frequencies for.
    - bases: list of bases to consider for k-mer generation (default is ['A', 'T', 'G', 'C']).
    - convert_freqs: whether to convert counts to frequencies by dividing by the number of k-mers in each sequence (default is True).
    - nonzero_only: whether to calculate frequencies only for k-mers that appear at least once in the dataset (default is False).

- make_kmer_dataframe: creates a DataFrame containing k-mer frequencies along with labels and counts for a given dataset.
    - ds: list of DataFrames for each round of the dataset.
    - k: length of the k-mers to calculate frequencies for.

- make_pca_dataframe: creates a DataFrame containing PCA-transformed k-mer frequencies along with labels and counts for a given dataset.
    - df: DataFrame containing k-mer frequencies and labels.
    - k: length of the k-mers to calculate frequencies for.
    - normalize_method: method for normalizing k-mer frequencies before PCA ('standard', 'center', or None).
    - return_pca: whether to return the PCA object along with the DataFrame (default is False).
    - use_precomputed_pca: whether to use a precomputed PCA object (default is False).
    - pca: precomputed PCA object to use if use_precomputed_pca is True (default is None).
    
- js_divergence_gaussians: calculates the Jensen-Shannon divergence between two multivariate Gaussian distributions, optionally weighted by counts.
    - X0: array of samples from the first distribution.
    - X1: array of samples from the second distribution.
    - w0: weights for the samples in X0 (default is None, which means equal weights).
    - w1: weights for the samples in X1 (default is None, which means equal weights).
    - base: logarithm base to use for the divergence calculation (default is 2).
    - n_samples: number of samples to use for estimating the divergence (default is 100000).
    - random_state: seed for reproducibility (default is None).

- make_distance_matrix: creates a distance matrix containing the Jensen-Shannon divergence between each pair of rounds in the dataset.
    - df: DataFrame containing PCA-transformed k-mer frequencies and labels for the dataset.
    - Rs: list of round labels corresponding to the rows in the DataFrame.
'''

def make_kmers(bases, k):
    return np.array([''.join(bp) for bp in itertools.product(bases, repeat = k)])

def normalize(X, method='standard'):
    if method == 'standard':
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    elif method == 'center':
        scaler = StandardScaler(with_std=False)
        X_scaled = scaler.fit_transform(X)
        X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
    elif method == None:
        X_scaled = X.copy()
    else:
        raise ValueError("Unknown normalization method.")

    return X_scaled

def cal_kmer_freqs(sequences, k, bases=['A', 'T', 'G', 'C'], convert_freqs=True, nonzero_only=False):

    lens = mylen(sequences) - k + 1
    N = len(sequences)
    base2int = np.zeros(256, dtype=np.uint8)
    for i, b in enumerate(bases):
        base2int[ord(b)] = i

    if nonzero_only:
        unique_kmers = set()
        for seq in sequences:
            if len(seq) < k:
                continue
            for i in range(len(seq) - k + 1):
                kmer = seq[i:i+k]
                unique_kmers.add(kmer)
        kmers = np.array(sorted(unique_kmers))
        kmer_to_idx = {kmer: idx for idx, kmer in enumerate(kmers)}
        result = np.zeros((N, len(kmers)), dtype=np.float32)

        for i, seq in enumerate(sequences):
            if len(seq) < k:
                continue
            for j in range(len(seq) - k + 1):
                kmer = seq[j:j+k]
                idx = kmer_to_idx[kmer]
                result[i, idx] += 1
    else:
        kmers = np.array([''.join(bp) for bp in itertools.product(bases, repeat = k)])
        result = np.zeros((N, len(kmers)), dtype=np.float32)
        factor = np.int32(len(bases) ** (k - 1))

        for i, seq in enumerate(sequences):
            if len(seq) < k:
                continue
            arr = base2int[np.frombuffer(seq.encode('ascii'), dtype=np.uint8)].astype(np.int32)

            idx = np.int32(0)
            for j in range(k):
                idx = idx * len(bases) + arr[j]
            result[i, idx] += 1

            for j in range(k, len(arr)):
                idx = (idx % factor) * len(bases) + arr[j]
                result[i, idx] += 1

    if convert_freqs:
        result /= lens[:, None]
    return DataFrame(data=result, columns=kmers)

def make_kmer_dataframe(ds, k):
    df = concatenate_dataframes(ds)
    labels = df['label']
    counts = df['counts']
    df = cal_kmer_freqs(df['seqs'], k)
    df = pd.concat([df, labels, counts], axis=1)
    return df

def make_pca_dataframe(df, k, normalize_method='standard', return_pca=False, use_precomputed_pca=False, pca=None):
    df_norm = normalize(df.iloc[:,:-2], method=normalize_method)

    if use_precomputed_pca and pca is not None:
        pca = pca
    elif use_precomputed_pca and pca is None:
        raise ValueError('pca must be provided when use_precomputed_pca is True.')
    else:
        pca = PCA()
        pca.fit(df_norm.values)
    contribution_ratios = pd.DataFrame(pca.explained_variance_ratio_)
    contribution_ratios = contribution_ratios.cumsum().values
    contribution_ratios = contribution_ratios.flatten()
    df_pca = DataFrame(data=pca.transform(df_norm.values), columns=[f"PC{i+1}" for i in range(len(pca.components_))])

    num_below_threshold = 3 * 4**(k-1)

    df_new = pd.concat([df_pca.loc[:,'PC1':f'PC{num_below_threshold}'], df['label'], df['counts']], axis=1)
    if return_pca:
        return df_new, pca
    else:
        return df_new

def js_divergence_gaussians(X0, X1, w0=None, w1=None, base=2, n_samples=100000, random_state=None):

    if w0 is not None:
        w0 = w0 / np.sum(w0)
        mu0 = np.average(X0, axis=0, weights=w0)
        X0_centered = X0 - mu0
        cov0 = np.cov(X0_centered.T, aweights=w0, bias=False)
    else:
        mu0 = np.mean(X0, axis=0)
        cov0 = np.cov(X0.T, bias=False)

    if w1 is not None:
        w1 = w1 / np.sum(w1)
        mu1 = np.average(X1, axis=0, weights=w1)
        X1_centered = X1 - mu1
        cov1 = np.cov(X1_centered.T, aweights=w1, bias=False)
    else:
        mu1 = np.mean(X1, axis=0)
        cov1 = np.cov(X1.T, bias=False)

    rng = np.random.default_rng(random_state)

    p = multivariate_normal(mean=mu0, cov=cov0)
    q = multivariate_normal(mean=mu1, cov=cov1)

    Xp = p.rvs(size=n_samples, random_state=rng)
    log_p_Xp = p.logpdf(Xp)
    log_q_Xp = q.logpdf(Xp)
    log_m_Xp = np.logaddexp(log_p_Xp, log_q_Xp) - np.log(2.0)

    KL_pm = np.mean(log_p_Xp - log_m_Xp)

    Xq = q.rvs(size=n_samples, random_state=rng)
    log_p_Xq = p.logpdf(Xq)
    log_q_Xq = q.logpdf(Xq)
    log_m_Xq = np.logaddexp(log_p_Xq, log_q_Xq) - np.log(2.0)

    KL_qm = np.mean(log_q_Xq - log_m_Xq)

    return 0.5 * (KL_pm + KL_qm) / np.log(base)

def make_distance_matrix(df, Rs):
    distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
    for i in range(len(Rs)):
        for j in range(i+1):
            X1 = df[df['label']==i].iloc[:,:-2].values
            w1 = df[df['label']==i]['counts'].values
            X2 = df[df['label']==j].iloc[:,:-2].values
            w2 = df[df['label']==j]['counts'].values
            distance_matrix[i][j] = js_divergence_gaussians(X1, X2, w1, w2)
            distance_matrix[j][i] = distance_matrix[i][j]
    return distance_matrix


'''
Functions for statistical analysis

- permutation_test: performs a permutation test to compare the average values of two groups of elements in a matrix
    - matrix: input square matrix to analyze.
    - N: number of permutations to perform (default is 100000).
    - order: number of diagonals to consider as the "close" group (default is 1).
    - side: which tail to consider for the p-value calculation ('right' for greater than observed difference, 'left' for less than observed difference, default is 'right').
    - pos: which part of the matrix to consider ('upper' for upper triangle, 'lower' for lower triangle, default is 'lower').
    - allow_diag: whether to include the diagonal elements in the "close" group (default is False).
'''

@njit(parallel=True)
def permutation_test(matrix, N=100000, order=1, side='right', pos='lower', allow_diag=False):
    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError("Input matrix must be square")
    
    if order >= matrix.shape[0]:
        raise ValueError("order must be less than the size of the matrix")

    if pos == 'upper':
        matrix = matrix.T
    elif pos == 'lower':
        pass
    else:
        raise ValueError("pos must be 'upper' or 'lower'")
    
    if side not in ['right', 'left']:
        raise ValueError("side must be 'right' or 'left'")
    
    matrix = np.tril(matrix, k=-1) + np.tril(matrix).T
    R = matrix.shape[0]
    CR_counts = order * (2 * R - order - 1) / 2
    NCR_counts = (R - order) * (R - order - 1) / 2
    k = -1
    
    if allow_diag:
        CR_counts += R
        k = 0
    
    CR_counts = np.float64(CR_counts)
    NCR_counts = np.float64(NCR_counts)

    NCR_ave = np.tril(matrix, k=-(order+1)).sum() / NCR_counts
    CR_ave = (np.tril(matrix, k=k) - np.tril(matrix, k=-(order+1))).sum() / CR_counts
    obs_diff = NCR_ave - CR_ave

    count_r, count_l = 0, 0
    for n in prange(N):
        perm1 = np.random.permutation(R)
        perm2 = np.random.permutation(R)
        if  not allow_diag:
            perm1 = perm2
        perm_matrix = matrix[perm1][:, perm2]

        NCR_ave_perm = np.tril(perm_matrix, k=-(order+1)).sum() / NCR_counts
        CR_ave_perm = (np.tril(perm_matrix, k=k) - np.tril(perm_matrix, k=-(order+1))).sum() / CR_counts
        perm_diff = NCR_ave_perm - CR_ave_perm

        count_r += (perm_diff > obs_diff)
        count_l += (perm_diff < obs_diff)
    p_val_r = (count_r + 1) / (N + 1)
    p_val_l = (count_l + 1) / (N + 1)

    if side == 'right':
        return p_val_r
    elif side == 'left':
        return p_val_l


'''
Functions for calculating contribution to heritability

- dist_consecutive_rounds: calculates the average distance between consecutive rounds in the distance matrix
    - distance_matrix: square matrix containing distances between rounds.
    - order: number of diagonals to consider as the "close" group (default is 1).

- dist_non_consecutive_rounds: calculates the average distance between non-consecutive rounds in the distance matrix
    - distance_matrix: square matrix containing distances between rounds.
    - order: number of diagonals to consider as the "close" group (default is 1).

- eval_index: calculates the difference between the average distance of non-consecutive rounds and consecutive rounds
    - distance_matrix: square matrix containing distances between rounds.
    - order: number of diagonals to consider as the "close" group (default is 1).

- rem_bases: removes a specified number of bases from either the 5' or 3' end of sequences in a DataFrame
    - df: input DataFrame.
    - end_from: string indicating which end to remove bases from ('5', '3', or 'none').
    - pos: position from which to start removing bases.
    - num_bases: number of bases to remove.

- shuffle_dataframe: shuffles the values in specified columns of a DataFrame while keeping the other columns unchanged
    - df: input DataFrame.
    - columns: list or tuple of column names to shuffle.
    - random_state: random state for reproducibility.
'''

def dist_consecutive_rounds(distance_matrix, order=1):
    consecutive_dists = []
    for i in range(len(distance_matrix)-1):
        for j in range(i+1, min(i+order+1, len(distance_matrix))):
            consecutive_dists.append(distance_matrix[i][j])

    return np.average(consecutive_dists) if consecutive_dists else 0

def dist_non_consecutive_rounds(distance_matrix, order=1):
    non_consecutive_dists = []
    for i in range(len(distance_matrix)-order-1):
        for j in range(i+1+order, len(distance_matrix)):
            non_consecutive_dists.append(distance_matrix[i][j])
    
    return np.average(non_consecutive_dists) if non_consecutive_dists else 0

def eval_index(distance_matrix, order=1):
    return dist_non_consecutive_rounds(distance_matrix, order=order) - dist_consecutive_rounds(distance_matrix, order=order)

def rem_bases(df, end_from, pos, num_bases):
    if end_from not in ['5', '3', 'none']:
        raise ValueError('end_from must be "5", "3", or "none".')
    if not (isinstance(pos, int) and pos >= 1):
        raise ValueError('pos must be an integer greater than or equal to 1.')
    if not (isinstance(num_bases, int) and num_bases >= 1):
        raise ValueError('num_bases must be an integer greater than or equal to 1.')
    
    if end_from == 'none':
        return df
    df = df.copy()
    seqs = df['seqs'].to_list()
    for i in range(len(seqs)):
        seq = seqs[i]
        if len(seq) < pos + num_bases - 1:
            continue
        if end_from == '5':
            new_seq = seq[:pos-1] + seq[pos+num_bases-1:]
        elif end_from == '3':
            if pos == 1:
                new_seq = seq[:-pos-num_bases+1]
            else:
                new_seq = seq[:-pos-num_bases+1] + seq[-pos+1:]
        seqs[i] = new_seq
    lens = mylen(seqs)
    df['lens'] = lens
    df['seqs'] = seqs
    return df

def shuffle_dataframe(df, columns, random_state=None):
    if not (isinstance(columns, (list, tuple, np.ndarray)) and all(col in df.columns for col in columns)):
        raise ValueError('columns must be a list or tuple of column names present in the DataFrame.')
    df_shuffled = df.copy()
    for column in columns:
        shuffled_values = df_shuffled[column].sample(frac=1, random_state=random_state).values
        df_shuffled[column] = shuffled_values
    return df_shuffled


'''
Functions for RNA folding and structure analysis

- rna_folding: calculates the minimum free energy (MFE) structure and its corresponding MFE for each sequence in a DataFrame using the ViennaRNA package.
    - df: input DataFrame containing a 'seqs' column with RNA sequences.
    - temp: temperature in Celsius to use for RNA folding (default is 22).

- make_distance_matrix_mfe: creates a distance matrix based on the Jensen-Shannon divergence of the MFE distributions for each round in the dataset.
    - df: DataFrame containing MFE values and labels for each sequence.
    - Rs: list of round labels corresponding to the rows in the DataFrame.
    - mfe_range: tuple specifying the range of MFE values to consider for the histogram bins (default is (-30, 0)).
    - step: step size for the histogram bins (default is 0.5).

- structure_labeling: labels each position in an RNA secondary structure string according to its structural context
    - structure: string representing the RNA secondary structure in dot-bracket notation
'''

def rna_folding(df, temp=22):
    sequences = df['seqs'].values
    md = RNA.md()
    md.temperature = temp

    structures = []
    mfes = []
    for seq in sequences:
        structure, mfe = RNA.fold_compound(seq, md).mfe()
        structures.append(structure)
        mfes.append(mfe)
    df['sts'] = structures
    df['mfes'] = mfes
    return df

def make_distance_matrix_mfe(df, Rs, mfe_range=(-30, 0), step=0.5):
    if not (isinstance(mfe_range, (list, tuple)) and len(mfe_range) == 2):
        raise ValueError('mfe_range must be a list or tuple of two numbers.')
    distance_matrix = np.full((len(Rs), len(Rs)), np.nan)
    for i in range(len(Rs)):
        p, _ = np.histogram(df[df['label']==i]['mfes'].values, weights=df[df['label']==i]['counts'],
                            bins=np.arange(mfe_range[0], mfe_range[1]+step, step))
        p = p / p.sum()
        for j in range(i+1):
            q, _ = np.histogram(df[df['label']==j]['mfes'].values, weights=df[df['label']==j]['counts'],
                                bins=np.arange(mfe_range[0], mfe_range[1]+step, step))
            q = q / q.sum()
            distance_matrix[i][j] = jensenshannon(p, q, base=2)
            distance_matrix[j][i] = distance_matrix[i][j]
    return distance_matrix

def structure_labeling(structure):
    structure_arr = ['E']*len(structure)
    # Left stem
    left_stem_pattern = r'\('
    for span in [m.span() for m in re.finditer(left_stem_pattern, structure)]:
        structure_arr[span[0]:span[1]] = ['L']*(span[1]-span[0])
    
    # Right stem
    right_stem_pattern = r'\)'
    for span in [m.span() for m in re.finditer(right_stem_pattern, structure)]:
        structure_arr[span[0]:span[1]] = ['R']*(span[1]-span[0])
    
    # Hairpin loop
    hairpin_pattern = r'\({1}\.+\){1}'
    for span in [m.span() for m in re.finditer(hairpin_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['H']*(span[1]-span[0]-2)
    
    # Internal loop / Bulge
    internal_pattern = r'\({1}\.+\({1}|\){1}\.+\){1}'
    for span in [m.span() for m in re.finditer(internal_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['I']*(span[1]-span[0]-2)
    
    # External loop
    external_pattern = r'\){1}\.+\({1}'
    for span in [m.span() for m in re.finditer(external_pattern, structure)]:
        structure_arr[span[0]+1:span[1]-1] = ['X']*(span[1]-span[0]-2)
    
    return ''.join(structure_arr)


'''
Functions for complementary sequence analysis
Usage of sub-functions are omitted for brevity, see comments and code for details.

- count_subseq_occurrences_sparse: main function to count the occurrences of k-mers in a list of sequences and return a sparse matrix.
    - seqs_sub: list of sub_sequences (k-mers) to count occurrences for.
    - seqs: list of original sequences to analyze for k-mer occurrences.
    - bases: list of bases to consider for encoding and counting k-mers (default is ['A', 'T', 'G', 'C']).
    - return_sparse: whether to return the counts as a sparse matrix (default is True).
    - expand_unique: whether to expand the unique k-mer counts to a full sparse representation (default is True).
    
- count_kmer_overlaps: main function to create a sparse matrix representing shared k-mer overlaps between two sets of sequences.
    - seqs1: list of first set of sequences.
    - seqs2: list of second set of sequences.
    - k: length of k-mers to consider.
    - bases: list of bases to consider for encoding and counting k-mers (default is ['A', 'T', 'G', 'C']).
    - return_sparse: whether to return the overlaps as a sparse matrix (default is True).
'''

### encodes a list of sequences into a 2D array of integers based on a provided mapping
def _encode_sequences(seqs, base2int):
    n = len(seqs)
    if n == 0:
        return np.empty((0, 0), dtype=np.int8), np.empty(0, dtype=np.int32)
    lens = np.fromiter((len(s) for s in seqs), dtype=np.int32, count=n)
    max_len = int(lens.max())
    arr = np.zeros((n, max_len), dtype=np.int8)
    for i, s in enumerate(seqs):
        if len(s) == 0:
            continue
        arr[i, :len(s)] = base2int[np.frombuffer(s.encode('ascii'), dtype=np.uint8)]
    return arr, lens

### encodes a list of k-mer sequences into their corresponding integer codes based on a provided mapping
def _encode_kmer_codes(seqs_sub, base2int, base):
    codes = np.zeros(len(seqs_sub), dtype=np.int32)
    for i, s in enumerate(seqs_sub):
        arr = base2int[np.frombuffer(s.encode('ascii'), dtype=np.uint8)].astype(np.int32)
        idx = np.int32(0)
        for v in arr:
            idx = idx * base + v
        codes[i] = idx
    return codes

### counts the occurrences of k-mers in encoded sequences using a target mapping and returns the data, row indices, and column indices for constructing a sparse matrix
@njit(parallel=True)
def _count_subseq_occurrences_numba(seqs_encoded, seqs_lens, k, base, factor, target_map, num_unique):
    n = seqs_encoded.shape[0]
    nnz_per_seq = np.zeros(n, dtype=np.int64)
    for i in prange(n):
        L = seqs_lens[i]
        if L < k:
            continue
        counts = np.zeros(num_unique, dtype=np.int32)
        idx = np.int32(0)
        for j in range(k):
            idx = idx * base + seqs_encoded[i, j]
        u = target_map[idx]
        if u >= 0:
            counts[u] += 1
        for j in range(k, L):
            idx = (idx % factor) * base + seqs_encoded[i, j]
            u = target_map[idx]
            if u >= 0:
                counts[u] += 1
        c = 0
        for u in range(num_unique):
            if counts[u] != 0:
                c += 1
        nnz_per_seq[i] = c
    total = np.sum(nnz_per_seq)
    rows = np.empty(total, dtype=np.int64)
    cols = np.empty(total, dtype=np.int64)
    data = np.empty(total, dtype=np.int32)
    offsets = np.zeros(n + 1, dtype=np.int64)
    for i in range(n):
        offsets[i + 1] = offsets[i] + nnz_per_seq[i]
    for i in prange(n):
        L = seqs_lens[i]
        if L < k:
            continue
        counts = np.zeros(num_unique, dtype=np.int32)
        idx = np.int32(0)
        for j in range(k):
            idx = idx * base + seqs_encoded[i, j]
        u = target_map[idx]
        if u >= 0:
            counts[u] += 1
        for j in range(k, L):
            idx = (idx % factor) * base + seqs_encoded[i, j]
            u = target_map[idx]
            if u >= 0:
                counts[u] += 1
        offset = offsets[i]
        t = 0
        for u in range(num_unique):
            c = counts[u]
            if c != 0:
                rows[offset + t] = u
                cols[offset + t] = i
                data[offset + t] = c
                t += 1
    return data, rows, cols

### expands a sparse representation of unique k-mer counts to a full sparse representation by replicating counts for sequences that share the same unique k-mers.
def _expand_unique_sparse(rows_u, cols_u, data_u, inverse_indices, num_unique):
    order = np.argsort(inverse_indices)
    counts = np.bincount(inverse_indices, minlength=num_unique)
    offsets = np.concatenate((np.array([0], dtype=np.int64), np.cumsum(counts)))
    entry_order = np.argsort(rows_u)
    rows_u = rows_u[entry_order]
    cols_u = cols_u[entry_order]
    data_u = data_u[entry_order]

    rows_list = []
    cols_list = []
    data_list = []
    pos = 0
    for u in range(num_unique):
        orig_rows = order[offsets[u]:offsets[u + 1]]
        if orig_rows.size == 0:
            continue
        start = pos
        while pos < rows_u.shape[0] and rows_u[pos] == u:
            pos += 1
        if start == pos:
            continue
        cols_slice = cols_u[start:pos]
        data_slice = data_u[start:pos]
        rows_block = np.repeat(orig_rows, cols_slice.shape[0])
        cols_block = np.tile(cols_slice, orig_rows.size)
        data_block = np.tile(data_slice, orig_rows.size)
        rows_list.append(rows_block)
        cols_list.append(cols_block)
        data_list.append(data_block)

    if not rows_list:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int32)
    rows = np.concatenate(rows_list)
    cols = np.concatenate(cols_list)
    data = np.concatenate(data_list)
    return data, rows, cols

def count_subseq_occurrences_sparse(seqs_sub, seqs, bases=('A', 'T', 'G', 'C'), return_sparse=True, expand_to_original=True):
    if len(seqs_sub) == 0:
        raise ValueError('seqs_sub is empty.')
    k = len(seqs_sub[0])
    if any(len(s) != k for s in seqs_sub):
        raise ValueError('All seqs_sub must have the same length k.')

    unique_subs, inverse_indices = np.unique(np.array(seqs_sub), return_inverse=True)
    num_unique = len(unique_subs)

    base2int = np.zeros(256, dtype=np.uint8)
    for i, b in enumerate(bases):
        base2int[ord(b)] = i
    base = np.int32(len(bases))
    factor = np.int32(base ** (k - 1))

    unique_codes = _encode_kmer_codes(unique_subs, base2int, base)
    target_map = np.full(base ** k, -1, dtype=np.int32)
    for u, code in enumerate(unique_codes):
        target_map[code] = u

    seqs_encoded, seqs_lens = _encode_sequences(seqs, base2int)
    data_u, rows_u, cols_u = _count_subseq_occurrences_numba(
        seqs_encoded, seqs_lens, k, base, factor, target_map, num_unique
    )

    if not expand_to_original:
        if return_sparse:
            S_unique = coo_matrix((data_u, (rows_u, cols_u)), shape=(num_unique, len(seqs)))
            return S_unique, unique_subs, inverse_indices
        return data_u, rows_u, cols_u, (num_unique, len(seqs)), unique_subs, inverse_indices

    data, rows, cols = _expand_unique_sparse(rows_u, cols_u, data_u, inverse_indices, num_unique)

    if return_sparse:
        return coo_matrix((data, (rows, cols)), shape=(len(seqs_sub), len(seqs)))
    return data, rows, cols, (len(seqs_sub), len(seqs))

### computes the overlap matrix between two sparse matrices represented in COO format, where the rows correspond to k-mers and the columns correspond to sequences.
@njit(parallel=True)
def _compute_overlap_matrix_numba(rows1, cols1, data1, rows2, cols2, data2, n1, n2, num_kmers):
    
    order1 = np.argsort(rows1)
    rows1_sorted = rows1[order1]
    cols1_sorted = cols1[order1]
    data1_sorted = data1[order1]
    
    order2 = np.argsort(rows2)
    rows2_sorted = rows2[order2]
    cols2_sorted = cols2[order2]
    data2_sorted = data2[order2]
    
    counts1 = np.zeros(num_kmers, dtype=np.int64)
    counts2 = np.zeros(num_kmers, dtype=np.int64)
    
    for r in rows1_sorted:
        counts1[r] += 1
    for r in rows2_sorted:
        counts2[r] += 1
    
    offsets1 = np.zeros(num_kmers + 1, dtype=np.int64)
    offsets2 = np.zeros(num_kmers + 1, dtype=np.int64)
    for i in range(num_kmers):
        offsets1[i + 1] = offsets1[i] + counts1[i]
        offsets2[i + 1] = offsets2[i] + counts2[i]
    
    total = np.int64(0)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        total += n1_entries * n2_entries
    
    result_rows = np.empty(total, dtype=np.int64)
    result_cols = np.empty(total, dtype=np.int64)
    result_data = np.empty(total, dtype=np.int32)
    
    kmer_offsets = np.zeros(num_kmers + 1, dtype=np.int64)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        kmer_offsets[kmer_idx + 1] = kmer_offsets[kmer_idx] + n1_entries * n2_entries
    
    for kmer_idx in prange(num_kmers):
        start1 = offsets1[kmer_idx]
        end1 = offsets1[kmer_idx + 1]
        start2 = offsets2[kmer_idx]
        end2 = offsets2[kmer_idx + 1]
        
        base_offset = kmer_offsets[kmer_idx]
        idx = 0
        for i in range(start1, end1):
            for j in range(start2, end2):
                result_rows[base_offset + idx] = cols1_sorted[i]
                result_cols[base_offset + idx] = cols2_sorted[j]
                result_data[base_offset + idx] = data1_sorted[i] * data2_sorted[j]
                idx += 1
    
    return result_data, result_rows, result_cols

### Binary version of the overlap matrix computation, where the data values are set to 1 for any overlap regardless of the original counts.
@njit(parallel=True)
def _compute_overlap_matrix_numba_binary(rows1, cols1, data1, rows2, cols2, data2, n1, n2, num_kmers):

    order1 = np.argsort(rows1)
    rows1_sorted = rows1[order1]
    cols1_sorted = cols1[order1]
    
    order2 = np.argsort(rows2)
    rows2_sorted = rows2[order2]
    cols2_sorted = cols2[order2]
    
    counts1 = np.zeros(num_kmers, dtype=np.int64)
    counts2 = np.zeros(num_kmers, dtype=np.int64)
    
    for r in rows1_sorted:
        counts1[r] += 1
    for r in rows2_sorted:
        counts2[r] += 1
    
    offsets1 = np.zeros(num_kmers + 1, dtype=np.int64)
    offsets2 = np.zeros(num_kmers + 1, dtype=np.int64)
    for i in range(num_kmers):
        offsets1[i + 1] = offsets1[i] + counts1[i]
        offsets2[i + 1] = offsets2[i] + counts2[i]
    
    total = np.int64(0)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        total += n1_entries * n2_entries
    
    result_rows = np.empty(total, dtype=np.int64)
    result_cols = np.empty(total, dtype=np.int64)
    result_data = np.ones(total, dtype=np.int32)
    
    kmer_offsets = np.zeros(num_kmers + 1, dtype=np.int64)
    for kmer_idx in range(num_kmers):
        n1_entries = counts1[kmer_idx]
        n2_entries = counts2[kmer_idx]
        kmer_offsets[kmer_idx + 1] = kmer_offsets[kmer_idx] + n1_entries * n2_entries
    
    for kmer_idx in prange(num_kmers):
        start1 = offsets1[kmer_idx]
        end1 = offsets1[kmer_idx + 1]
        start2 = offsets2[kmer_idx]
        end2 = offsets2[kmer_idx + 1]
        
        base_offset = kmer_offsets[kmer_idx]
        idx = 0
        for i in range(start1, end1):
            for j in range(start2, end2):
                result_rows[base_offset + idx] = cols1_sorted[i]
                result_cols[base_offset + idx] = cols2_sorted[j]
                idx += 1
    
    return result_data, result_rows, result_cols

def count_kmer_overlaps(seqs1, seqs2, k, bases=('A', 'T', 'G', 'C'), return_sparse=True, binary=False):
    if len(seqs1) == 0 or len(seqs2) == 0:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    lens1 = np.fromiter((len(s) for s in seqs1), dtype=np.int32, count=len(seqs1))
    lens2 = np.fromiter((len(s) for s in seqs2), dtype=np.int32, count=len(seqs2))
    if np.any(lens1 < k) or np.any(lens2 < k):
        raise ValueError('All sequences must have length at least k.')
    
    kmers1 = set()
    for s in seqs1:
        for i in range(len(s) - k + 1):
            kmers1.add(s[i:i + k])
    
    kmers2 = set()
    for s in seqs2:
        for i in range(len(s) - k + 1):
            kmers2.add(s[i:i + k])
    
    common_kmers = list(kmers1.intersection(kmers2))
    
    if not common_kmers:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    data1, rows1, cols1, shape1, unique_kmers, _ = count_subseq_occurrences_sparse(
        common_kmers, seqs1, bases=bases, return_sparse=False, expand_to_original=False
    )
    data2, rows2, cols2, shape2, _, _ = count_subseq_occurrences_sparse(
        common_kmers, seqs2, bases=bases, return_sparse=False, expand_to_original=False
    )
    
    if len(data1) == 0 or len(data2) == 0:
        if return_sparse:
            return coo_matrix((len(seqs1), len(seqs2)))
        return np.empty(0, dtype=np.int32), np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64), (len(seqs1), len(seqs2))
    
    if binary:
        result_data, result_rows, result_cols = _compute_overlap_matrix_numba_binary(
            rows1, cols1, data1, rows2, cols2, data2, len(seqs1), len(seqs2), shape1[0]
        )
    else:
        result_data, result_rows, result_cols = _compute_overlap_matrix_numba(
            rows1, cols1, data1, rows2, cols2, data2, len(seqs1), len(seqs2), shape1[0]
        )

    if return_sparse:
        if not binary:
            S = coo_matrix((result_data, (result_rows, result_cols)), shape=(len(seqs1), len(seqs2)))
            S.sum_duplicates()
        if binary:
            hashes = result_rows * len(seqs2) + result_cols
            unique_hashes, counts = np.unique(hashes, return_counts=True)
            data = np.ones_like(counts, dtype=np.int32)
            S = coo_matrix((data, (unique_hashes // len(seqs2), unique_hashes % len(seqs2))), shape=(len(seqs1), len(seqs2)))
        return S
    
    return result_data, result_rows, result_cols, (len(seqs1), len(seqs2))

In [ ]:
### Fig1C & S6: Length distribution of sequences in each round

name = 'N14'
min_len = 15
max_len = 45

ds, Rs = dataloader(name, length_range=(min_len, max_len), method='none')
length_counts = np.zeros((max_len - min_len + 1, len(Rs)))
for i in range(len(Rs)):
    lens = ds[i]['lens'].values
    counts = ds[i]['counts'].values
    length_counts[:, i] = [counts[lens == l].sum() for l in range(min_len, max_len + 1)]

np.save(f'data/processed/length_distribution/{name}_length_distribution_no-sampling_m{min_len}_M{max_len}.npy', length_counts)

In [ ]:
### Fig2AB & S11A: Frequencies of individual nucleotides relative to those in the initial round

name = 'N14'
N = 10

base_freqs_set = []
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, length_range='default', N=100000, remove_rounds='default')
    base_freqs = np.zeros((4, len(Rs)))
    for i in range(len(Rs)):
        d = ds[i]
        d_kmer = cal_kmer_freqs(d['seqs'], k=1, convert_freqs=False, nonzero_only=False)
        total_counts = (d['counts']*d['lens']).sum()
        for j, base in enumerate(['A', 'T', 'G', 'C']):
            base_freqs[j, i] = (d_kmer[base] * d['counts']).sum() / total_counts
    base_freqs_set.append(base_freqs)
    print(f'Elapsed time for iteration {n+1}: {time.time() - start:.2f} seconds')
base_freqs_set = np.array(base_freqs_set)
np.save(f'data/processed/base_composition/{name}_base_composition.npy', base_freqs_set)

In [ ]:
### Fig3AB & S11BC: Round-by-round comparison of JS divergence based on 3-mer frequency vectors between populations at any two rounds.

name = 'N14'
k = 3
N = 10

distance_matrix_set = []
for _ in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=100000, remove_rounds='default', method='all')
    df = make_kmer_dataframe(ds, k)
    df_new = make_pca_dataframe(df, k)
    distance_matrix = make_distance_matrix(df_new, Rs)
    distance_matrix_set.append(distance_matrix)
    print(f'Elapsed time for {name}, n={_+1}: {time.time()-start:.2f}s')

np.save(f'data/processed/distance_matrix/{name}_k{k}_distance_matrix.npy', np.array(distance_matrix_set))

In [ ]:
### Fig 3D & S11D: Position-dependent contributions of 3-mers to the observed heritable traits.

name = 'N14'
N = 10
k = 3

end_from = '5'
num_bases = 3
pos_max = 14

eval_indices = np.zeros((N, pos_max+1))
for i in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=100000)
    df = make_kmer_dataframe(ds, k)
    df_new, pca = make_pca_dataframe(df, k, return_pca=True)
    distance_matrix = make_distance_matrix(df_new, Rs)
    eval_index_val = eval_index(distance_matrix)

    eval_indices[i, 0] = eval_index_val

    for pos in range(1, pos_max+1):
        ds_rem = [rem_bases(d, end_from, pos, num_bases) for d in ds]
        df = make_kmer_dataframe(ds_rem, k)
        df_new = make_pca_dataframe(df, k, use_precomputed_pca=True, pca=pca)
        distance_matrix = make_distance_matrix(df_new, Rs)
        eval_index_val = eval_index(distance_matrix)
        eval_indices[i, pos] = eval_index_val
    print(f'Elapsed time for {name}, n={i+1}: {time.time()-start:.2f}s')

np.save(open(f'data/processed/pos_contribution/{name}_positional_contribution_{end_from}_n{num_bases}_k{k}.npy', 'wb'), eval_indices)

In [ ]:
### Fig 4AB: Round-by-round comparison of JS divergence based on ΔGMFE distributions between populations at any two rounds

name = 'N14'
N = 10
threshold = 0
mfe_range = (-30, 0)

distance_matrix_set = []

for _ in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=100000)
    df = concatenate_dataframes(ds)
    df = rna_folding(df)
    df = df[df['mfes']!=0].reset_index(drop=True)
    distance_matrix = make_distance_matrix_mfe(df, Rs, mfe_range=mfe_range)
    distance_matrix_set.append(distance_matrix)
    print(f'Elapsed time for {name}, n={_+1}: {time.time()-start:.2f}s')
np.save(f'data/processed/structure/{name}_distance_matrix.npy', np.array(distance_matrix_set))

In [ ]:
### Fig 5BCD & S12: Round-by-round comparison of the number of complementary 6-nt pairs between UT (or T or O) and O populations.

name = 'N14'
k = 8
N = 10
sample_size = 100000
terminal = True
unpaired = True
if not terminal:
    unpaired = False

sub_len_5 = k
sub_len_3 = k

mat_nnz_set = []
for _ in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=sample_size)

    seqs_list = []
    seqs_comp_list = []
    for d in ds:
        d = deploy_df_by_counts(d)
        seqs = d['seqs'].values

        if terminal:
            if unpaired:
                d = rna_folding(d)
                unpair_5 = [st[:sub_len_5].count('.') for st in d['sts'].values]
                unpair_3 = [st[-sub_len_3:].count('.') for st in d['sts'].values]
                valid_inds = np.where((np.array(unpair_5) == sub_len_5) | (np.array(unpair_3) == sub_len_3))[0]
                seqs_comp = []
                for idx in valid_inds:
                    seq = seqs[idx]
                    if (unpair_5[idx] == sub_len_5) & (unpair_3[idx] == sub_len_3):
                        seq = seq[:sub_len_5] + seq[-sub_len_3:]
                    elif unpair_5[idx] == sub_len_5:
                        seq = seq[:sub_len_5]
                    elif unpair_3[idx] == sub_len_3:
                        seq = seq[-sub_len_3:]
                    seq = seq.translate(str.maketrans('ATGC', 'TACG'))[::-1]
                    seqs_comp.append(seq)
                seqs_list.append(seqs)
                seqs_comp_list.append(seqs_comp)
            else:
                seqs_comp = []
                for seq in seqs:
                    seq = seq[:sub_len_5] + seq[-sub_len_3:]
                    seq = seq.translate(str.maketrans('ATGC', 'TACG'))[::-1]
                    seqs_comp.append(seq)
                seqs_list.append(seqs)
                seqs_comp_list.append(seqs_comp)
        else:
            seqs_comp = [seq.translate(str.maketrans('ATGC', 'TACG'))[::-1] for seq in seqs]
            seqs_list.append(seqs)
            seqs_comp_list.append(seqs_comp)

    mat_nnz = np.zeros((len(Rs), len(Rs)))

    for i in range(len(Rs)):
        for j in range(i+1):
        # for j in range(len(Rs)):
            S = count_kmer_overlaps(seqs_list[i], seqs_comp_list[j], k, return_sparse=True, binary=True)
            mat_nnz[i, j] = S.nnz
            if i != j:
                mat_nnz[j, i] = S.nnz
    mat_nnz_set.append(mat_nnz)
    print(f'Elapsed time for {name} iteration {len(mat_nnz_set)}: {time.time() - start:.4f} seconds')

mat_nnz_set = np.array(mat_nnz_set)
result = {
    'mat_nnz': mat_nnz_set,
    'sub_len_5': sub_len_5,
    'sub_len_3': sub_len_3,
    'k': k,
    'N': N,
    'Rs': Rs,
    'unpaired': unpaired,
    'terminal': terminal,
    'sample_size': sample_size
}
np.savez_compressed(f'data/processed/overlap/{name}_overlap_summary_sub{sub_len_5}_{sub_len_3}_k{k}_n{N}_unpaired{unpaired}_terminal{terminal}_sm.npz', **result)

In [ ]:
### Fig 6B & 13B: Frequencies of XX(NNXX)n motifs detected after incubating the N14>p pool in the presence of AG(NNAG)5 or CU(NNCU)5, relative to those in the absence of semi-random RNAs.

name = '2mer'
max_repeat = 4
min_repeat = 1
N = 10
sampling_num = 100000

freqs_set = []
kmers = make_kmers(['A', 'T', 'G', 'C'], 2)
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name=name, N=sampling_num, remove_rounds='default')
    df = concatenate_dataframes(ds)
    seqs = df['seqs'].values
    freqs = np.zeros((len(Rs), 16, max_repeat-min_repeat+1))
    for i, kmer in enumerate(kmers):
        for r in range(min_repeat, max_repeat+1):
            pattern = kmer+(r'.{2}'+kmer)*r
            inds = [j for j, seq in enumerate(seqs) if re.search(pattern, seq)]
            df_sub = df.iloc[inds].reset_index(drop=True)
            freq = [df_sub[df_sub['label']==l]['counts'].sum()/df[df['label']==l]['counts'].sum() for l in range(len(Rs))]
            freqs[:, i, r-min_repeat] = freq
    freqs_set.append(freqs)
    print(f'Elapsed time for iteration {n+1}: {time.time()-start:.2f}s')

# Pseudo-counts
freqs_set = np.array(freqs_set)
freqs_set += 1/sampling_num
np.save(f'data/processed/2mer/2mer_repeat_freqs.npy', freqs_set)

In [ ]:
### FigS2: Identification of potential sequence contaminants

name = 'N14'

with open(f'data/filtered/Series/{name}_filtered.pkl', 'rb') as handle:
    ds = pickle.load(handle)
subprocess.run(['mkdir', f'data/fasta/{name}'], stdout=subprocess.DEVNULL)
for i in range(len(ds)):
    df = ds[i][:1000]
    seqs = df['seqs'].values
    make_fasta(seqs, f'data/fasta/{name}/{name}_{i+1}', key='DNA')

In [ ]:
### Before running the next section, BLAST fasta files created in above section against the genomic database, and save the results in data/tsv/{name}/

In [ ]:
### FigS2: Identification of potential sequence contaminants

name = 'N14'
threshold = 90
Rs = [1, 2, 3, 5, 6, 8, 10, 11, 13, 15, 17, 19, 20, 22, 24, 26, 28, 29, 30]
length_range = (18, 45)

with open(f'data/filtered/Series/{name}_filtered.pkl', 'rb') as handle:
    ds = pickle.load(handle)

ds = [d[(d['lens']>=length_range[0]) & (d['lens']<=length_range[1])].reset_index(drop=True) for d in ds]
ds = [d[:1000] for d in ds]


top_nums = [10, 20, 50, 100, 200, 500, 1000]
contamination_probs = []
for i in range(len(ds)):
    df = ds[i]
    df_blast = read_blast_results(f'data/tsv/{name}/{name}_{i+1}')
    df = eval_blast_result(df, df_blast)

    df.columns = ['seqs', 'counts', 'lens', 'qcover_pre', 'ident_pre']
    seqs = df['seqs'].to_list()
    df_sub = df[:200]
    contami_seqs = df_sub[(df_sub['qcover_pre']>=threshold) & (df_sub['ident_pre']>=threshold)]['seqs'].to_list()
    run_blast(seqs, contami_seqs, 'blast_result', num_threads=4, use_precomputed_db=False, rm_db=True)
    df_blast_contami = read_blast_results('blast_result')
    df = eval_blast_result(df, df_blast_contami)

    probs = []
    for top_num in top_nums:
        df_top = df[:top_num]
        df_filtered = df_top[(df_top['qcover']<threshold) | (df_top['ident']<threshold)]
        if len(df_filtered) > 0:
            df_filtered = df_filtered[(df_filtered['qcover_pre']>=threshold) & (df_filtered['ident_pre']>=threshold)]
        else:
            df_filtered = df_top.iloc[0:0]

        probs.append(len(df_filtered) / len(df_top))
    contamination_probs.append(probs)
contamination_probs = np.array(contamination_probs)
np.save(f'data/processed/contamination_probs/{name}_contamination_probs_blast.npy', contamination_probs)

In [ ]:
### Fig S3: Cumulative explained variance of PCA for k-mer frequency vectors.

N = 10
names = ['N14']
ks = np.arange(1, 5)

ccr_set = []
for k in ks:
    start = time.time()
    for name in names:
        ccrs = []
        for _ in range(N):
            ds, _ = dataloader(name, N=100000)
            df = make_kmer_dataframe(ds, k=k)
            df_normalized = normalize(df.iloc[:,:-2])

            pca = PCA()
            pca.fit(df_normalized.values)
            contribution_ratios = pd.DataFrame(pca.explained_variance_ratio_)
            contribution_ratios = contribution_ratios.cumsum().values
            ccrs.append(contribution_ratios)
        contribution_ratios = np.mean(ccrs, axis=0)
        ccr_set.append(contribution_ratios.flatten())
    print(f'Elapsed time for k={k}: {time.time() - start:.2f} seconds')

ccr_set = np.array(ccr_set, dtype=object)
np.save('data/processed/pca_contribution_ratio/pca_contribution_ratios.npy', ccr_set)

In [ ]:
### Fig S4: Squared Mahalanobis distances of the first 3 × 4^(k-1) principal component scores for sequences in the initial round.

names = ['N14']
k = 3

mhdists_list = []
counts_list = []
for i, name in enumerate(names):
    start = time.time()
    ds, Rs = dataloader(name, N=100000)
    df = make_kmer_dataframe(ds, k=k)
    df_new = make_pca_dataframe(df, k=k)
    X = df_new[df_new['label']==0].iloc[:,:-2].values
    # X = normalize(X).values
    mean = np.mean(X, axis=0)
    cov = np.cov(X.T)
    cov_inv = np.linalg.inv(cov)
    n = X.shape[1]
    mhdists = np.array([distance.mahalanobis(x, mean, cov_inv)**2 for x in X])
    mhdists_list.append(mhdists)
    counts_list.append(df_new[df_new['label']==0]['counts'].values)
    print(f'Elapsed time for {name}: {time.time() - start:.2f} seconds')

mhdists_list = np.array(mhdists_list, dtype=object)
counts_list = np.array(counts_list, dtype=object)
np.save(f'data/processed/mahalanobis/mhdists_list_k{k}.npy', mhdists_list)
np.save(f'data/processed/mahalanobis/counts_list_k{k}.npy', counts_list)

In [ ]:
### Fig S7: Sequence similarity among populations from different rounds.

name = 'N14'
key = 'genotype'
N = 10

start = time.time()
matrix_set = []
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, remove_rounds='default', method='all', N=100000)
    matrix = np.zeros((len(Rs), len(Rs)))
    for i in range(len(Rs)):
        for j in range(i):
            di = ds[i]
            dj = ds[j]
            if key == 'cosine_similarity':
                di['counts'] = di['counts'].values / di['counts'].sum()
                dj['counts'] = dj['counts'].values / dj['counts'].sum()
                df = concatenate_dataframes([di, dj])
                df = pd.pivot_table(df, index='seqs', columns='label', values='counts', fill_value=0)
                vec_i = df.iloc[:, 0].values
                vec_j = df.iloc[:, 1].values
                matrix[i, j] = vec_i.dot(vec_j) / (np.linalg.norm(vec_i) * np.linalg.norm(vec_j))
                del df, vec_i, vec_j
            elif key == 'count':
                seqsi = set(di['seqs'].values)
                seqsj = set(dj['seqs'].values)
                common_seqs = seqsi.intersection(seqsj)
                count = sum([min(di[di['seqs']==s]['counts'].values[0],
                                dj[dj['seqs']==s]['counts'].values[0]) for s in common_seqs])
                matrix[i, j] = count
            elif key == 'genotype':
                seqsi = list(di['seqs'].values)
                seqsj = list(dj['seqs'].values)
                seqs_all, counts = zip(*collections.Counter(seqsi + seqsj).most_common())
                counts = np.array(counts)
                matrix[i, j] = len(np.where(counts==2)[0])
            matrix[j, i] = matrix[i, j]
        if key == 'cosine_similarity':
            matrix[i, i] = 1.0
        elif key == 'count':
            matrix[i, i] = ds[i]['counts'].sum()
        elif key == 'genotype':
            matrix[i, i] = len(ds[i])
        matrix_set.append(matrix)
    print(f'Elapsed time for {name}, n={n+1}: {time.time()-start:.2f}s')
matrix_set = np.array(matrix_set)
np.save(f'data/processed/geno_similarity/{name}_{key}_matrix_set.npy', matrix_set)
del ds

In [ ]:
### Fig S8: Sequence diversity of each RNA pool over rounds.

name = 'N14'
N = 10

entropy_values_set = []
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, length_range='default', N=100000, remove_rounds='default')
    entropy_values = np.zeros(len(Rs))
    for i in range(len(Rs)):
        freqs = ds[i]['counts'] / ds[i]['counts'].sum()
        entropy = -np.sum(freqs * np.log2(freqs)) / np.log2(len(freqs))
        entropy_values[i] = entropy
    entropy_values_set.append(entropy_values)
    print(f'Elapsed time for iteration {n+1}: {time.time() - start:.2f} seconds')
entropy_values_set = np.array(entropy_values_set)
np.save(f'data/processed/entropy/{name}_entropy.npy', entropy_values_set)

In [ ]:
### Fig S9: Contributions of individual 3-mers to the observed heritability.

name = 'N14'
k = 3
N = 10

dists = np.zeros((N, 4**k + 1))
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=100000)
    df = make_kmer_dataframe(ds, k)
    df_new, pca = make_pca_dataframe(df, k, return_pca=True)
    distance_matrix = make_distance_matrix(df_new, Rs)
    dists[n][0] = eval_index(distance_matrix)

    kmers = make_kmers(['A', 'T', 'G', 'C'], k)
    for i, kmer in enumerate(kmers):
        df_shuffled = shuffle_dataframe(df, [kmer])
        df_shuffled_new = make_pca_dataframe(df_shuffled, k, use_precomputed_pca=True, pca=pca)
        distance_matrix_shuffled = make_distance_matrix(df_shuffled_new, Rs)
        dists[n][i+1] = eval_index(distance_matrix_shuffled)
    print(f'Elapsed time for {name}, n={n+1}: {time.time()-start:.2f}s')

np.save(f'data/processed/kmer_contribution/{name}_kmer_contribution.npy', dists)

In [ ]:
### Fig S10: The probabilities of nucleotides assigned to each of the three categories were calculated for each position from the 5′ end and the 3′ end.

name = 'N14'
N = 10
min_len = 18
max_len = 30


bp_freqs_set_5 = []
bp_freqs_set_3 = []
for n in range(N):
    start = time.time()
    ds, Rs = dataloader(name, N=10000, remove_rounds='default', length_range=(min_len, max_len))
    df = concatenate_dataframes(ds)
    df = rna_folding(df)
    sts = df['sts'].values
    sts = [structure_labeling(s) for s in sts]
    sts_aligned_3 = [list('-'* (max_len - len(s)) + s) for s in sts]
    sts_aligned_5 = [list(s + '-'* (max_len - len(s))) for s in sts]
    sts_aligned_3 = np.array(sts_aligned_3, dtype=object)
    sts_aligned_5 = np.array(sts_aligned_5, dtype=object)
    total_count = df['counts'].sum()
    counts = df['counts'].values
    bp_freqs_3 = np.zeros((3, sts_aligned_3.shape[1]))
    bp_freqs_5 = np.zeros((3, sts_aligned_5.shape[1]))
    for i, labels in enumerate([('E'), ('H', 'X', 'I'), ('L', 'R')]):
        for pos in range(sts_aligned_3.shape[1]):
            bp_freqs_3[i, pos] = counts[np.isin(sts_aligned_3[:, pos], labels)].sum() / total_count
            bp_freqs_5[i, pos] = counts[np.isin(sts_aligned_5[:, pos], labels)].sum() / total_count
    bp_freqs_set_3.append(bp_freqs_3)
    bp_freqs_set_5.append(bp_freqs_5)
    print(f'Elapsed time for {name} iteration {n+1}: {time.time() - start:.2f} seconds')
bp_freqs_set_3 = np.array(bp_freqs_set_3)
bp_freqs_set_5 = np.array(bp_freqs_set_5)
np.save(f'data/processed/base_pairing/{name}_bp_freqs_3.npy', bp_freqs_set_3)
np.save(f'data/processed/base_pairing/{name}_bp_freqs_5.npy', bp_freqs_set_5)

In [ ]:
### Fig S13A: Distribution of predicted minimum free energies (ΔGMFE) of fully random 22-nt RNAs and the two designed semi-random 22-nt RNAs.

def generate_random_sequences(N, length, base=['A', 'T', 'G', 'C'], base_freqs=None, random_state=None):
    if random_state is not None:
        random.seed(random_state)
    if base_freqs is not None:
        base_freqs = np.array(base_freqs)
        shape = base_freqs.shape
        if len(shape) > 2:
            raise ValueError("base_freqs should be a 1D or 2D array.")
        elif shape[0] != len(base):
            raise ValueError("Length of axis 0 of base_freqs should match the length of base.")
        elif len(shape) == 2 and shape[1] != length:
            raise ValueError("Length of axis 1 of base_freqs should match the specified length.")
        if len(shape) == 1:
            base_freqs = np.tile(base_freqs[:, np.newaxis], (1, length))
        base_freqs = base_freqs / base_freqs.sum(axis=0)
    else:
        base_freqs = np.full((len(base), length), 1/len(base))
    
    sequences = []
    for _ in range(N):
        sequence = []
        for i in range(length):
            probs = base_freqs[:, i]
            b = random.choices(base, weights=probs, k=1)[0]
            sequence.append(b)
        sequences.append(''.join(sequence))
    return sequences

def apply_random_base(seq, random_state=None):
    if random_state is not None:
        random.seed(random_state)
    bases = ['A', 'U', 'G', 'C']
    return ''.join(random.choice(bases) if base == 'N' else base for base in seq)


fixed_seq1 = 'AGNNAGNNAGNNAGNNAGNNAG'
fixed_seq2 = 'CUNNCUNNCUNNCUNNCUNNCU'
N = 100000
temp = 22.0


random_seqs = generate_random_sequences(N=N, length=len(fixed_seq1), base=['A', 'U', 'G', 'C'], base_freqs=None)
fixed_seqs1 = [apply_random_base(fixed_seq1) for _ in range(N)]
fixed_seqs2 = [apply_random_base(fixed_seq2) for _ in range(N)]

md = RNA.md()
md.temperature = temp

mfes_random = np.array([RNA.fold_compound(seq, md).mfe()[1] for seq in random_seqs])
mfes_fixed1 = np.array([RNA.fold_compound(seq, md).mfe()[1] for seq in fixed_seqs1])
mfes_fixed2 = np.array([RNA.fold_compound(seq, md).mfe()[1] for seq in fixed_seqs2])

bin_width = 0.5
bins = np.arange(-10.0, 1.0, bin_width)
freqs_random, _ = np.histogram(mfes_random, bins=bins)
freqs_random = freqs_random / freqs_random.sum()
freqs_fixed1, _ = np.histogram(mfes_fixed1, bins=bins)
freqs_fixed2, _ = np.histogram(mfes_fixed2, bins=bins)
freqs_fixed1 = freqs_fixed1 / freqs_fixed1.sum()
freqs_fixed2 = freqs_fixed2 / freqs_fixed2.sum()
bin_centers = (bins[:-1] + bins[1:]) / 2

result = {
    'bin_centers': bin_centers,
    'freqs_random': freqs_random,
    'freqs_fixed1': freqs_fixed1,
    'freqs_fixed2': freqs_fixed2,
    'temp': temp,
    'N': N,
    'fixed_seq1': fixed_seq1,
    'fixed_seq2': fixed_seq2
}
np.savez_compressed(f'data/processed/overlap/mfe_distribution_fixed_seqs.npz', **result)